# Zero- and Few-Shot Coordinate Extraction

This notebook evaluates zero-shot and few-shot prompting strategies using large language models (via the OpenAI API) for extracting and normalizing geographic coordinates from encyclopedic entries.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import instructor
from typing import List
from pydantic import BaseModel, Field
from sklearn.model_selection import KFold
import pandas as pd
import numpy as np
from tqdm import tqdm

In [ ]:
# Get the OPENAI_API_KEY from the .env file
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

client = instructor.from_openai(OpenAI(api_key=api_key))

In [ ]:
# Data structure
class DataModel(BaseModel):
    coords: List[str] = Field(description="List of geographic coordinates in DMS format (e.g., [\"43 55' 44\" N 19 49' E\"])")

examples = '''Exemple 1 : 
    INPUT: "* ALBI, (Géog.) ville de France, capitale de l'Albigeois, dans le haut Languedoc : elle est sur le Tarn. Long. 19. 49. lat. 43. 55. 44."
    OUTPUT:{["43 55' 44\" N 19 49' E"]}
    ---
    Exemple 2 : 
    INPUT: "JORGIANE, (Géog.) riviere d'Asie dans la Perse, qui donne son nom à une ville qu'elle arrose, & se décharge dans la mer Caspienne, à 89d de long. & à 38 de latit. La ville de son nom qu'elle baigne est dans la Corassane. Long. 85. latit. 37. (D. J.)"
    OUTPUT:{["38 N 89 E", "37 N 85 E"]}
---
'''

user_preprompt = f'''
Ta tâche est d’extraire les coordonnées géographiques indiquées dans le texte et de les réécrire uniquement sous une forme normalisée, en respectant la syntaxe du format Degrés/Minutes/Secondes (DMS). 
Format de sortie attendu : une liste de chaînes de caractères, chaque chaîne représentant une coordonnée avec latitude et longitude, dans le format suivant :  
"DEGRE MIN' SEC" N/S DEGRE MIN' SEC" E/O"
Règles importantes :
- Ne pas convertir ni interpréter les valeurs (exemple : ne pas transformer 309° en 51° Ouest).
- Ne pas ajouter de minutes ou secondes si elles ne sont pas présentes dans le texte original.
- Ne pas ajouter de zéros implicites (si le texte ne donne que des degrés, garder uniquement les degrés).
- Toujours indiquer la direction (N/S pour la latitude, E/O pour la longitude) en fonction de l'hémisphère.
- Si plusieurs coordonnées sont présentes (pour plusieurs localisations), les séparer par des virgules dans la liste.
- Si plusieurs coordonnées sont présentes (pour une même localisation), le premier élément de la liste doit être égale à la chaîne de caractères "alt".
- Il est possible que le texte ne contienne que la latitude ou que la longitude.
- Simplement reprendre les nombres et unités données dans le texte et les réécrire avec la syntaxe normalisée DMS.
Voici quelques exemples pour illustrer le format attendu :
{examples}
INPUT: 
'''

def run_gpt(user_prompt, client, model_name, data_model, max_retries=3):
    
    response = client.chat.completions.create(
        model=model_name,
        max_retries=max_retries,
        messages=[
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        response_model=data_model,
    )
    return " | ".join(response.coords)

In [2]:
model_name = 'gpt-5-mini'

In [ ]:
# test examples

ex1 = "* AACH ou ACH, s. f. petite ville d'Allemagne dans le cercle de Souabe, près de la source de l'Aach. Long. 26. 57. lat. 47. 55."
ex2 = "* ABNAKIS, s. m. Peuple de l'Amérique septentrionale, dans le Canada. Il occupe le 309. de long. & le 46. de lat."

out1 = run_gpt(user_preprompt + ex1, client, model_name, DataModel)
out2 = run_gpt(user_preprompt + ex2, client, model_name, DataModel)

print(out1)
print(out2)

In [ ]:
df = pd.read_json("../edda_coordinata.json")
df.head()

In [ ]:
def normalize_coords(val):
    if not isinstance(val, list):
        return val.replace("[[\'", '').replace('[["', '').replace("\']]", '').replace('"]]', '').replace("], [", ' | ').replace("\'", "'").replace('\\', '')
    return " | ".join([" ".join(inner) for inner in val])

df["coordinates_txt"] = df["coordinates"].apply(normalize_coords)
df.head()

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
dfs = []
for fold, (train_idx, val_idx) in enumerate(kf.split(df)):
    print(f"Fold {fold + 1}")
    mask = df.index.isin(val_idx)
    val_dataset = df[mask]
    print(f"  Validation set size: {len(val_dataset)}")
    dfs.append(val_dataset)

In [ ]:
# run_gpt(user_preprompt + ex1, client, model_name, DataModel) over the dataframe

for fold in range(5):
    tqdm.pandas(desc="Prediction des types d'articles")
    dfs[fold]["predicted_coordinates"] = dfs[fold]["text"].progress_apply(lambda x: run_gpt(user_preprompt + x, client, model_name, DataModel))
    dfs[fold]

In [ ]:
for i, d in enumerate(dfs):
    d.to_csv(f"../outputs/{model_name}/edda_coordinates_fold{i}.csv", index=False)

In [3]:
dfs = [pd.read_csv(f"../outputs/{model_name}/edda_coordinates_fold{i}.csv") for i in range(5)]

In [4]:
from Levenshtein import distance

def average_cer(references, hypotheses):
    assert len(references) == len(hypotheses), "Mismatched list lengths."
    total_edits = 0
    total_chars = 0
    for ref, hyp in zip(references, hypotheses):
        if ref is None or hyp is None:
            continue  # skip invalid pairs
        ref = str(ref)
        hyp = str(hyp)
        if len(ref) == 0:
            continue  # avoid div by zero
        total_edits += distance(ref, hyp)
        total_chars += len(ref)
    return total_edits / total_chars if total_chars > 0 else float('inf')

In [5]:
# evaluation
e_matches = []
cer_scores = []
missing_data_counts = 0
for i in range(len(dfs)):
    d = dfs[i]
    d["gpt_preds"] = d["gpt_preds"].fillna("")
    missing_data_counts += len(d[d["coordinates_txt"].isna() | d["gpt_preds"].isna()])
    d["coordinates_txt"] = d["coordinates_txt"].str.replace("alt | ", "", regex=False)

    # Exact Match
    exact_matches = [int(p == l) for p, l in zip(d["gpt_preds"], d["coordinates_txt"])]
    em_score = np.mean(exact_matches)
    print(f"  Exact Match: {em_score:.4f}")
    # store exact matches
    e_matches.extend(exact_matches)

    # Character Error Rate
    cer_score = average_cer(d["coordinates_txt"], d["gpt_preds"])
    print(f"  Char Error Rate: {cer_score:.4f}")
    # store CER scores
    cer_scores.append(cer_score)

print(f"Overall Exact Match: {np.mean(e_matches):.4f}")
print(f"Micro-average CER: {np.mean(cer_scores):.4f}")


  Exact Match: 0.8644
  Char Error Rate: 0.0337
  Exact Match: 0.8571
  Char Error Rate: 0.0280
  Exact Match: 0.8415
  Char Error Rate: 0.0368
  Exact Match: 0.8300
  Char Error Rate: 0.0326
  Exact Match: 0.8539
  Char Error Rate: 0.0218
Overall Exact Match: 0.8494
Micro-average CER: 0.0306
